# Model Definition and Evaluation
## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)


In [18]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
# Import models you're considering

import optuna
import torchmetrics
import copy

## Model Selection

[Discuss the type(s) of models you consider for this task, and justify the selection.]



## Feature Engineering

[Describe any additional feature engineering you've performed beyond what was done for the baseline model.]


In [27]:
# Load the dataset
# For feature engineering see the script "data_engineering_features.ipynb"
df = pd.read_csv("../hanabi_dataset_features.csv")

# Perform any feature engineering steps
# Feature and target variable selection
X = df.iloc[:, 2:]
y = df['final_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)

df.head()

,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,...,white_count,blue_count,red_count,p1_hand_sum,p1_avg_value,p2_hand_sum,p2_avg_value,p1_unique_cards,p2_unique_cards,avg_max_possible_progress
0,gamec4a276e8bbb31378.log,10,0,0,0,0,0,0,0,0,...,1,3,4,17,3.4,13,2.6,4,5,0.2
1,game146052cdf75868b1.log,21,0,0,0,0,0,0,0,0,...,2,3,4,15,3.0,10,2.0,5,5,1.2
2,gamea89f449023dbd46f.log,8,0,0,0,0,0,0,0,0,...,1,3,3,12,2.4,12,2.4,4,4,0.4
3,gamee78abd094f6f5d73.log,14,0,0,0,0,0,0,0,0,...,2,3,3,14,2.8,12,2.4,4,4,0.6
4,game53b40ed46697f583.log,6,0,0,0,0,0,0,0,0,...,2,2,4,13,2.6,13,2.6,5,5,1.4


## Hyperparameter Tuning

[Discuss any hyperparameter tuning methods you've applied, such as Grid Search or Random Search, and the rationale behind them.]


In [20]:
# Implement hyperparameter tuning
# Example using GridSearchCV with a DecisionTreeClassifier
# param_grid = {'max_depth': [2, 4, 6, 8]}
# grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5)
# grid_search.fit(X_train, y_train)

def define_search_space(trial):
    parameters = {}

    # Hyperparameter for model architecture
    # control the complexity and capacity of the neural network
    parameters["num_hidden_layers"] = trial.suggest_int(
        "num_hidden_layers",
        2,
        4
    )

    parameters["hidden_size"] = trial.suggest_categorical(
    "hidden_size",
    [32, 64, 128, 256]
    )

    # Hyperparameter for Regularization
    # help prevent overfitting
    parameters["dropout_rate"] = trial.suggest_float(
        "dropout_rate",
        0.0,
        0.5
    )

    parameters["weight_decay"] = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-2,
        log=True
    )

    # Hyperparameter for optimization
    # control how the network learns
    parameters["learning_rate"] = trial.suggest_float(
        "learning_rate",
        1e-5,
        1e-2,
        log=True
    )

    parameters["batch_size"] = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64, 128]
    )

    return parameters

# Dynamic model definition
class HanabiDynamic(nn.Module):
    def __init__(
        self,
        input_dim,
        num_hidden_layers,
        hidden_size,
        dropout_rate
    ):
        super().__init__()

        layers = []

        current_dim = input_dim

        for _ in range(num_hidden_layers):

            layers.append(
                nn.Linear(current_dim, hidden_size)
            )

            layers.append(nn.ReLU())

            layers.append(
                nn.Dropout(dropout_rate)
            )

            current_dim = hidden_size

        layers.append(
            nn.Linear(current_dim, 1)
        )

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Define the objective function for optuna
def objective(trial):

    params = define_search_space(trial)

    # Create model
    model = HanabiDynamic(
        input_dim=X_train.shape[1],
        num_hidden_layers=params["num_hidden_layers"],
        hidden_size=params["hidden_size"],
        dropout_rate=params["dropout_rate"]
    )

    # Optimizer
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"]
    )

    criterion = nn.MSELoss()

    # Create dataloaders
    X_train_tensor = torch.FloatTensor(X_train.values)
    y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

    X_val_tensor = torch.FloatTensor(X_val.values)
    y_val_tensor = torch.FloatTensor(y_val.values).reshape(-1, 1)

    train_dataset = torch.utils.data.TensorDataset(
        X_train_tensor,
        y_train_tensor
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=params["batch_size"],
        shuffle=True
    )

    # Training
    model.train()

    for epoch in range(10):

        for x_batch, y_batch in train_loader:

            optimizer.zero_grad()

            predictions = model(x_batch)

            loss = criterion(
                predictions,
                y_batch
            )

            loss.backward()

            optimizer.step()

    # Validation
    model.eval()

    with torch.no_grad():

        val_predictions = model(X_val_tensor)

        val_loss = criterion(
            val_predictions,
            y_val_tensor
        )

    return val_loss.item()


In [21]:
# Run optima on the defined search space
study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=30
)

print("The best model parameters:", study.best_params)
print("The best MSE:", study.best_value)
print("The best RMSE:", np.sqrt(study.best_value))

# assign the parameters to variables to use later on
num_hidden_layers = study.best_params["num_hidden_layers"]
hidden_size = study.best_params["hidden_size"]
dropout_rate = study.best_params["dropout_rate"]
learning_rate = study.best_params["learning_rate"]
weight_decay = study.best_params["weight_decay"]
batch_size = study.best_params["batch_size"]

[I 2026-06-15 20:27:34,665] A new study created in memory with name: no-name-b2926582-b68b-4333-872e-65eff7f532e5
[I 2026-06-15 20:27:35,078] Trial 0 finished with value: 168.6037139892578 and parameters: {'num_hidden_layers': 3, 'hidden_size': 32, 'dropout_rate': 0.4982201115565072, 'weight_decay': 1.0242656559129477e-06, 'learning_rate': 1.746214977788651e-05, 'batch_size': 32}. Best is trial 0 with value: 168.6037139892578.
[I 2026-06-15 20:27:35,301] Trial 1 finished with value: 36.89297103881836 and parameters: {'num_hidden_layers': 3, 'hidden_size': 128, 'dropout_rate': 0.15561378969946266, 'weight_decay': 0.0029689372338270134, 'learning_rate': 0.00017036337663000036, 'batch_size': 128}. Best is trial 1 with value: 36.89297103881836.
[I 2026-06-15 20:27:35,697] Trial 2 finished with value: 33.95534133911133 and parameters: {'num_hidden_layers': 2, 'hidden_size': 128, 'dropout_rate': 0.44243570198094007, 'weight_decay': 0.0012619466699937572, 'learning_rate': 0.000298977361408591

The best model parameters: {'num_hidden_layers': 3, 'hidden_size': 64, 'dropout_rate': 0.10669448143034399, 'weight_decay': 0.0018785007514810954, 'learning_rate': 0.004011425440909556, 'batch_size': 128}
The best MSE: 32.63130569458008
The best RMSE: 5.712381788236854


## Implementation

[Implement the final model(s) you've selected based on the above steps.]


In [22]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

#only transform the validation and test sets, no fitting -> avoiding data leakage
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [23]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [24]:
#class HanabiScorePredictor(nn.Module):
#    def __init__(self, input_dim):
#        super(HanabiScorePredictor, self).__init__()
#        self.network = nn.Sequential(
#            nn.Linear(input_dim, 128),
#            nn.ReLU(),
#            nn.Dropout(0.2),
#            nn.Linear(128, 64),
#            nn.ReLU(),
#            nn.Linear(64, 32),
#            nn.ReLU(),
#            nn.Dropout(0.2),
#            nn.Linear(32, 1)
#        )
#        
#    def forward(self, x):
#        return self.network(x)

#model = HanabiScorePredictor(input_dim= X_train.shape[1])
model = HanabiDynamic(
    input_dim=X_train.shape[1],
    num_hidden_layers=num_hidden_layers,
    hidden_size=hidden_size,
    dropout_rate=dropout_rate
)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

In [25]:
num_epochs = 100

train_losses = []
val_losses = []

# early stopping settings
patience = 10
best_val_loss = float("inf")
epochs_without_improvement = 0
best_model_state = None

# learning rate scheduler 
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

for epoch in range(num_epochs):

    # Training
    model.train()
    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)
            val_loss = criterion(outputs, y_batch)

            running_val_loss += val_loss.item()

    epoch_val_loss = running_val_loss / len(val_loader)
    val_losses.append(epoch_val_loss)

    # LR scheduler
    scheduler.step(epoch_val_loss)

    # early stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_without_improvement = 0

        # save best model
        best_model_state = copy.deepcopy(model.state_dict())

    else:
        epochs_without_improvement += 1

    # Logging
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

    # Stop training early
    if epochs_without_improvement >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

# Restore the best model
model.load_state_dict(best_model_state)

Epoch [1/100] Train Loss: 161.9090 Val Loss: 110.0778
Epoch [2/100] Train Loss: 62.4413 Val Loss: 42.2468
Epoch [3/100] Train Loss: 37.1855 Val Loss: 36.8043
Epoch [4/100] Train Loss: 33.7777 Val Loss: 35.6140
Epoch [5/100] Train Loss: 34.4978 Val Loss: 34.2839
Epoch [6/100] Train Loss: 33.2395 Val Loss: 34.0952
Epoch [7/100] Train Loss: 32.1028 Val Loss: 34.1438
Epoch [8/100] Train Loss: 31.0863 Val Loss: 34.0378
Epoch [9/100] Train Loss: 33.8190 Val Loss: 34.0686
Epoch [10/100] Train Loss: 34.4624 Val Loss: 34.2859
Epoch [11/100] Train Loss: 28.6862 Val Loss: 34.6824
Epoch [12/100] Train Loss: 31.7658 Val Loss: 36.0659
Epoch [13/100] Train Loss: 30.9030 Val Loss: 34.7778
Epoch [14/100] Train Loss: 32.7751 Val Loss: 34.7936
Epoch [15/100] Train Loss: 30.7706 Val Loss: 34.7680
Epoch [16/100] Train Loss: 31.2565 Val Loss: 34.8967
Epoch [17/100] Train Loss: 30.6385 Val Loss: 35.0260
Epoch [18/100] Train Loss: 34.1458 Val Loss: 34.9761
Early stopping triggered at epoch 18


<All keys matched successfully>

In [26]:
model.eval()

with torch.no_grad():
    predictions = model(X_test_tensor)

    mse = nn.MSELoss()(predictions, y_test_tensor)
    rmse = torch.sqrt(mse)

print(f"Test MSE: {mse.item():.4f}")
print(f"Test RMSE: {rmse.item():.4f}")

Test MSE: 31.5882
Test RMSE: 5.6203


## Evaluation Metrics

[Clearly specify which metrics you'll use to evaluate the model performance, and why you've chosen these metrics.]


In [45]:
# Evaluate the model using your chosen metrics
# Example for classification
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))

# Example for regression
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here


## Comparative Analysis

[Compare the performance of your model(s) against the baseline model. Discuss any improvements or setbacks and the reasons behind them.]


In [46]:
# Comparative Analysis code (if applicable)
# Example: comparing accuracy of the baseline model and the new model
# print(f"Baseline Model Accuracy: {baseline_accuracy}, New Model Accuracy: {new_model_accuracy}")
